In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the Week 2 dataset
csv_path = Path("../../north_sales.csv")
df = pd.read_csv(csv_path)

# Display the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values per column:")
print(df.isnull().sum())

Dataset shape: (3, 7)

Column names:
['id', 'product', 'region', 'amount', 'sale_date', 'quantity', 'discount']

First 5 rows:


,id,product,region,amount,sale_date,quantity,discount
0,1,Laptop,North,85000,2026-01-15,2,0.05
1,5,Phone,North,45000,2026-02-05,4,0.05
2,7,Tablet,North,32000,2026-04-01,2,0.08



Missing values per column:
id           0
product      0
region       0
amount       0
sale_date    0
quantity     0
discount     0
dtype: int64


In [2]:
# Six Data Quality Dimensions

quality_checks = {
    "Completeness": df.notna().all().all(),
    "Accuracy": (
        (df["amount"] >= 0).all()
        and (df["quantity"] > 0).all()
    ),
    "Consistency": df["id"].is_unique,
    "Timeliness": pd.to_datetime(
        df["sale_date"], errors="coerce"
    ).notna().all(),
    "Uniqueness": df["id"].is_unique,
    "Validity": (
        df["discount"].between(0, 1).all()
        and df["product"].notna().all()
        and df["region"].notna().all()
    ),
}

for dimension, passed in quality_checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{dimension}: {status}")

Completeness: PASS
Accuracy: PASS
Consistency: PASS
Timeliness: PASS
Uniqueness: PASS
Validity: PASS


In [3]:
%pip install great_expectations


  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
     ---------------------------------------- 0.0/133.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/133.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/133.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/133.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/133.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/133.0 kB ? eta -:--:--
     --- ------------------------------------ 10.2/133.0 kB ? eta -:--:--
     --- ------------------------------------ 10.2/133.0 kB ? eta -:--:--
     -------- ---------------------------- 30.7/133.0 kB 259.2 kB/s eta 0:00:01
     ----------- ------------------------- 41.0/133.0 kB 245.8 kB/s eta 0:00:01
     ----------------- ------------------- 61.4/133.0 kB 272.3 kB/s eta 0:00:01
     ----------------- ------------------- 61.4/133.0 kB 272.3 kB/s eta 0:00:01
     -------------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import great_expectations as gx

print("Great Expectations version:", gx.__version__)

Great Expectations version: 1.24.0


In [5]:
# Eight data quality expectations

expectations = {
    "1. ID should be unique": df["id"].is_unique,
    "2. ID should not be missing": df["id"].notna().all(),
    "3. Product should not be missing": df["product"].notna().all(),
    "4. Region should not be missing": df["region"].notna().all(),
    "5. Amount should be non-negative": (df["amount"] >= 0).all(),
    "6. Quantity should be positive": (df["quantity"] > 0).all(),
    "7. Discount should be between 0 and 1": df["discount"].between(0, 1).all(),
    "8. Sale date should be valid": pd.to_datetime(
        df["sale_date"], errors="coerce"
    ).notna().all(),
}

for name, passed in expectations.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

1. ID should be unique: PASS
2. ID should not be missing: PASS
3. Product should not be missing: PASS
4. Region should not be missing: PASS
5. Amount should be non-negative: PASS
6. Quantity should be positive: PASS
7. Discount should be between 0 and 1: PASS
8. Sale date should be valid: PASS


In [6]:
# Make a copy so the original dataset stays unchanged
df_test = df.copy()

# Introduce 3 intentional data-quality errors
df_test.loc[0, "amount"] = -500
df_test.loc[1, "discount"] = 1.5
df_test.loc[2, "sale_date"] = "invalid-date"

print("Original dataset remains unchanged:")
display(df.head())

print("\nDataset with intentional errors:")
display(df_test)

Original dataset remains unchanged:


,id,product,region,amount,sale_date,quantity,discount
0,1,Laptop,North,85000,2026-01-15,2,0.05
1,5,Phone,North,45000,2026-02-05,4,0.05
2,7,Tablet,North,32000,2026-04-01,2,0.08



Dataset with intentional errors:


,id,product,region,amount,sale_date,quantity,discount
0,1,Laptop,North,-500,2026-01-15,2,0.05
1,5,Phone,North,45000,2026-02-05,4,1.50
2,7,Tablet,North,32000,invalid-date,2,0.08


In [7]:
# Validate the dataset containing intentional errors

error_checks = {
    "ID should be unique": df_test["id"].is_unique,
    "ID should not be missing": df_test["id"].notna().all(),
    "Product should not be missing": df_test["product"].notna().all(),
    "Region should not be missing": df_test["region"].notna().all(),
    "Amount should be non-negative": (df_test["amount"] >= 0).all(),
    "Quantity should be positive": (df_test["quantity"] > 0).all(),
    "Discount should be between 0 and 1": df_test["discount"].between(0, 1).all(),
    "Sale date should be valid": pd.to_datetime(
        df_test["sale_date"], errors="coerce"
    ).notna().all(),
}

for name, passed in error_checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{name}: {status}")

ID should be unique: PASS
ID should not be missing: PASS
Product should not be missing: PASS
Region should not be missing: PASS
Amount should be non-negative: FAIL
Quantity should be positive: PASS
Discount should be between 0 and 1: FAIL
Sale date should be valid: FAIL
